# 06B GROMACS Dry Polymer

This notebook converts the GAFF2 AMBER files to a self-contained dry GROMACS polymer workflow. It focuses on conversion, topology validation, box sanity checks, and quick local minimisation.

It does not solvate the system and it does not execute HPC jobs.


## Workflow Scope

- Engine: GROMACS
- System: dry polymer
- Input: `gaff2/<SYSTEM>.prmtop` and `gaff2/<SYSTEM>.inpcrd`
- Main output: `examples/output/md_tests/<SYSTEM>/gromacs/dry_polymer/`
- Optional seeded folders: `gromacs/solvated_polymer/` and `gromacs/charmm_gui_membrane/` are kept separate for later notebooks

The CHARMM-GUI-style membrane ladder remains an advanced optional workflow and is not part of the default PHA oligomer validation path.


## Workflow Comparison: Simplified Polymer vs CHARMM-GUI-Style

This project keeps two GROMACS equilibration workflows side by side. They are not interchangeable defaults; they serve different scientific and engineering purposes.

| Workflow | Default location | Stages | Best use | Main advantage | Main tradeoff |
| --- | --- | --- | --- | --- | --- |
| Simplified polymer workflow | `gromacs/solvated_polymer/` | minimisation -> NVT -> NPT -> production | polymer benchmarking, rapid iteration, small PHA oligomer tests, method development | fast to understand, fast to run, fewer moving parts | less conservative for fragile heterogeneous systems |
| CHARMM-GUI-style workflow | `gromacs/charmm_gui_membrane/` | minimisation -> multiple restrained/relaxation equilibration stages -> production | membrane proteins, enzyme/polymer complexes, large assemblies, sensitive interfaces | gradual relaxation reduces shock to complex systems | longer setup, more stages to inspect, more runtime |

For polymer-only benchmarking, the simplified workflow is the default because a small solvated PHA oligomer does not usually need a long staged relaxation ladder. For membrane proteins, enzyme/polymer systems, aggregation systems, or any system where interfaces and packing are delicate, the CHARMM-GUI-style staged workflow is the safer template.

Both workflows remain in the project. The folder separation is intentional so running or editing one route does not overwrite the other.


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd() if (Path.cwd() / "src" / "iphasimulator").exists() else Path.cwd().parent
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

output_root = repo_root / "examples" / "output"
system_name = "PHB4"
md_root = output_root / "md_tests" / system_name
print(f"Repository: {repo_root}")
print(f"System: {system_name}")
print(f"MD output root: {md_root}")

gaff2_dir = md_root / "gaff2"
gromacs_dir = md_root / "gromacs"
dry_polymer_dir = gromacs_dir / "dry_polymer"
prmtop_path = gaff2_dir / f"{system_name}.prmtop"
inpcrd_path = gaff2_dir / f"{system_name}.inpcrd"

{
    "prmtop": prmtop_path,
    "prmtop_exists": prmtop_path.exists(),
    "inpcrd": inpcrd_path,
    "inpcrd_exists": inpcrd_path.exists(),
    "dry_polymer_dir": dry_polymer_dir,
}

## Prepare Converted GROMACS Inputs

This writes the dry polymer `.gro`, `.top`, index file, MDP template, and local minimisation script. The workflow folders are separate so rerunning one route does not overwrite another route.


In [ ]:
from iphasimulator.simulation.gromacs_runner import prepare_gromacs_run_folder

RUN_GROMACS_CONVERSION = False

if RUN_GROMACS_CONVERSION:
    prepared = prepare_gromacs_run_folder(
        prmtop_path,
        inpcrd_path,
        gromacs_dir,
        system_name,
        workflow_type="polymer",
    )
    prepared
else:
    print("Set RUN_GROMACS_CONVERSION = True to regenerate the dry GROMACS workflow folder.")

## Validate Converted Topology

This checks that every `#include` in the dry topology resolves. ParmEd often writes standalone topologies, which is also valid.


In [ ]:
from iphasimulator.simulation.gromacs_runner import validate_gromacs_run_folder

if (dry_polymer_dir / "topol.top").exists():
    topology_validation = validate_gromacs_run_folder(dry_polymer_dir)
    print(f"Standalone topology: {topology_validation.is_standalone}")
    print(f"Missing includes: {topology_validation.missing_files}")
    topology_validation
else:
    print(f"Missing dry topology: {dry_polymer_dir / 'topol.top'}")

## Check Dry Minimisation Inputs

The dry minimisation script must be run from inside `gromacs/dry_polymer/`.


In [ ]:
from iphasimulator.simulation.gromacs_runner import check_gromacs_minimization_inputs

minimization_check = check_gromacs_minimization_inputs(dry_polymer_dir)
print(f"Ready: {minimization_check.ready}")
print(f"Command: {minimization_check.command_text}")
for path in minimization_check.missing_files:
    print(f"Missing: {path}")

## Optional Local Dry Minimisation

This is disabled by default. It writes GROMACS outputs inside `gromacs/dry_polymer/`.


In [ ]:
from iphasimulator.simulation.gromacs_runner import run_gromacs_local_minimization

RUN_DRY_GROMACS_MINIMIZATION = False

if RUN_DRY_GROMACS_MINIMIZATION:
    result = run_gromacs_local_minimization(dry_polymer_dir)
    print(result.stdout)
else:
    print("Set RUN_DRY_GROMACS_MINIMIZATION = True to run dry local minimisation.")